# 05 · Find hidden themes (topic modeling)

**The problem:** the CFPB form makes customers pick an issue from a fixed list. Real complaints are messier:
"my money was frozen after a Zelle payment" doesn't fit neatly into any box. Topic modeling groups complaints
by the words they share, **without being told the categories**, so themes nobody had a label for can show up.

**Method:** NMF (non-negative matrix factorization) on TF-IDF. In plain words: it finds a small set of
"word recipes" (topics) such that every complaint is a mix of a few recipes. Each topic is a list of words with
weights, and each complaint gets a weight for each topic.

In [1]:
from radar import setup, run_sql, save_table
setup()

'Local: tables go to data/complaint_radar.duckdb'

### Load the complaints

In [2]:
data = run_sql("SELECT complaint_id, date_received, product, issue, text, narrative FROM complaints")
print(f"{len(data):,} complaints")

26,558 complaints


### Copy-paste letters
Some complaints are the same letter sent many times (often from credit repair templates). If we kept every copy,
a single template would become its own "topic". So the model learns from one copy of each letter: we treat
complaints whose first 200 characters are identical as copies. Every complaint still gets a topic afterwards.

In [3]:
data["opening"] = data["text"].str.slice(0, 200)
copies = data.groupby("opening")["complaint_id"].transform("count")
print(f"{(copies > 1).sum():,} complaints share their opening with another complaint")
unique = data.drop_duplicates("opening")
print(f"{len(unique):,} distinct letters to learn the topics from")

5,025 complaints share their opening with another complaint
22,030 distinct letters to learn the topics from


### Words to ignore
Besides normal English stop words ("the", "and"), we drop two kinds of words:
* words every complaint letter uses ("called", "told", "tried")
* words every **financial** complaint uses ("bank", "account", "payment", "credit"), which would otherwise pull
  very different problems into one big "money" topic

What's left are the words that tell problems apart: "escrow", "repossession", "overdraft", "identity theft".

In [4]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

LETTER_WORDS = {"told", "called", "call", "said", "asked", "company", "received", "time", "day", "days", "did", "know",
                "just", "like", "want", "need", "got", "going", "make", "made", "able", "help", "contacted", "contact",
                "phone", "email", "letter", "sent", "number", "information", "amount", "consumer", "cfpb", "complaint",
                "please", "thank", "regarding", "stated", "says", "dont", "didnt", "im", "ive", "month", "months", "year",
                "years", "today", "week", "weeks", "also", "would", "could", "still", "never", "even", "one", "two", "get",
                "tried", "times", "send", "went", "trying", "saying", "person", "don", "getting", "despite", "issue",
                "resolve", "request", "requesting", "multiple", "review", "explanation"}
MONEY_WORDS = {"bank", "card", "credit", "account", "accounts", "money", "loan", "payment", "payments", "report",
               "pay", "paid", "funds", "balance", "usc", "section", "act", "law", "fcra", "fdcpa"}
STOP = list(ENGLISH_STOP_WORDS | LETTER_WORDS | MONEY_WORDS)

vectorizer = TfidfVectorizer(stop_words=STOP, ngram_range=(1, 2), min_df=15, max_df=0.25, max_features=30_000, sublinear_tf=True)
X = vectorizer.fit_transform(unique["text"])
X.shape

(22030, 13855)

### How many topics?
There's no single right number. Too few and different problems merge (car loans and mortgages become one
"loans" topic); too many and one problem splits into fragments. We try 12, 16 and 20, look at how much the top
words of different topics overlap (lower is better: topics are more distinct), and how evenly complaints spread.

In [5]:
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF

words = np.array(vectorizer.get_feature_names_out())

def overlap(model, n=12):
    tops = [set(words[np.argsort(row)[::-1][:n]]) for row in model.components_]
    return float(np.mean([len(a & b) / n for i, a in enumerate(tops) for b in tops[i + 1:]]))

tries = []
for k in [12, 16, 20]:
    model = NMF(n_components=k, init="nndsvda", random_state=42, max_iter=500)
    sizes = np.bincount(model.fit_transform(X).argmax(axis=1), minlength=k)
    tries.append({"topics": k, "avg_word_overlap": round(overlap(model), 4), "biggest_topic_share_pct": round(100 * sizes.max() / sizes.sum(), 1)})
data_tries = pd.DataFrame(tries)
data_tries

,topics,avg_word_overlap,biggest_topic_share_pct
0,12,0.0215,25.3
1,16,0.0139,25.5
2,20,0.0110,17.3


### Fit the final model
We use 20 topics: at 12 and 16, separate problems (car loans and late fees, for example) were still merged.

In [6]:
K = 20
nmf = NMF(n_components=K, init="nndsvda", random_state=42, max_iter=600)
nmf.fit(X)
W = nmf.transform(vectorizer.transform(data["text"]))     # topic weights for every complaint, copies included
print(f"{K} topics")

20 topics


### Give every complaint its main topic
A complaint's main topic is the one with the highest weight. Complaints with almost no weight anywhere
(very generic letters) are marked as topic -1, "unclear".

In [7]:
data["topic"] = W.argmax(axis=1)
data["topic_weight"] = W.max(axis=1)
data.loc[data["topic_weight"] < 0.01, "topic"] = -1
data["topic"].value_counts().sort_index()

topic
-1       86
 0      991
 1     1440
 2     2834
 3      849
 4     1699
 5     1875
 6      259
 7     1411
 8      378
 9      519
 10    1233
 11     577
 12    3891
 13    1210
 14    1398
 15    2320
 16    1471
 17    1303
 18     327
 19     487
Name: count, dtype: int64

### Describe each topic
For each topic: its top words, how many complaints it holds, which product those complaints come from, and how
many are copies of the same letter (a high share means a copy-paste campaign rather than many separate stories).

In [8]:
rows = []
for t in range(K):
    members = data[data["topic"] == t]
    product_mix = members["product"].value_counts(normalize=True).head(3)
    rows.append({
        "topic": t,
        "top_words": ", ".join(words[np.argsort(nmf.components_[t])[::-1][:12]]),
        "complaints": len(members),
        "share_pct": round(100 * len(members) / len(data), 1),
        "main_products": "; ".join(f"{p} {v:.0%}" for p, v in product_mix.items()),
        "copy_pct": round(100 * (copies[members.index] > 1).mean(), 1) if len(members) else 0.0,  # share that are copy-paste letters
        "top_issue": members["issue"].mode()[0] if len(members) else None,
    })
topics = pd.DataFrame(rows)
data_topics = topics.sort_values("complaints", ascending=False)
data_topics

,topic,top_words,complaints,share_pct,main_products,copy_pct,top_issue
12,12,"customer, service, customer service, online, u...",3891,14.7,Credit card 26%; Bank account 22%; Money trans...,3.0,Managing an account
2,2,"reporting, inaccurate, fair reporting, fair, a...",2834,10.7,Credit reporting 54%; Debt collection 29%; Cre...,38.7,Incorrect information on your report
15,15,"dispute, transaction, transactions, claim, den...",2320,8.7,Credit card 35%; Bank account 34%; Money trans...,0.9,Problem with a purchase shown on your statement
5,5,"debt, collection, validation, debt collection,...",1875,7.1,Debt collection 78%; Credit reporting 11%; Cre...,15.1,Attempts to collect debt not owed
4,4,"identity, theft, identity theft, fraudulent, v...",1699,6.4,Credit reporting 56%; Debt collection 18%; Cre...,33.4,Incorrect information on your report
16,16,"fee, charged, fees, charge, charges, statement...",1471,5.5,Credit card 46%; Bank account 14%; Personal lo...,0.1,Fees or interest
1,1,"unfair, unprotected, vulnerable unprotected, f...",1440,5.4,Money transfer 99%; Bank account 0%; Credit re...,83.0,Other transaction problem
7,7,"mortgage, escrow, insurance, home, property, t...",1411,5.3,Mortgage 95%; Debt collection 2%; Personal loa...,0.1,Trouble during payment process
14,14,"car, vehicle, dealership, repossession, auto, ...",1398,5.3,Vehicle loan 77%; Personal loan 7%; Debt colle...,0.0,Repossession
17,17,"check, deposit, deposited, closed, capital, di...",1303,4.9,Bank account 66%; Money transfer 9%; Credit ca...,4.6,Managing an account


### The most typical complaints of each topic
The complaints with the highest weight for a topic are its clearest examples. We keep 5 per topic: notebook 06
shows them to a language model to name and summarise the topic, and the website shows a few.

In [9]:
examples = []
for t in range(K):
    best = np.argsort(W[:, t])[::-1][:5]
    for rank, i in enumerate(best):
        examples.append({"topic": t, "rank": rank + 1, "complaint_id": data.iloc[i]["complaint_id"],
                         "product": data.iloc[i]["product"], "narrative": data.iloc[i]["narrative"]})
data_examples = pd.DataFrame(examples)
data_examples.head()

,topic,rank,complaint_id,product,narrative
0,0,1,23886217,Mortgage,FORMAL COMPLAINT REGARDING MORTGAGE SERVICING ...
1,0,2,18939066,Vehicle loan,"To Whom It May Concern, I am writing to addres..."
2,0,3,21854865,Mortgage,Summary of Issue Since initiating my refinance...
3,0,4,12223962,Credit card,"Description of the Issue : On XX/XX/XXXX, my w..."
4,0,5,17417529,Credit reporting,"CFPB Escalation Letter 45- days, Non- Response..."


### Topic trends over time
How each topic's share of the sample changes month by month (through July 2026, the last full month).

In [10]:
data["month"] = pd.to_datetime(data["date_received"]).dt.to_period("M").dt.to_timestamp()
trend = data[(data["topic"] >= 0) & (data["month"] < "2026-08-01")]
trend = trend.groupby(["month", "topic"]).size().rename("complaints").reset_index()
trend["share_pct"] = (100 * trend["complaints"] / trend.groupby("month")["complaints"].transform("sum")).round(2)
trend.tail()

,month,topic,complaints,share_pct
365,2026-07-01,15,81,14.11
366,2026-07-01,16,45,7.84
367,2026-07-01,17,46,8.01
368,2026-07-01,18,5,0.87
369,2026-07-01,19,5,0.87


### Save everything as tables

In [11]:
save_table(topics, "topics")
save_table(data_examples, "topic_examples")
save_table(trend, "topic_trend")
save_table(data[["complaint_id", "topic", "topic_weight"]], "complaint_topics")

'complaint_topics: 26,558 rows'